# Valuation for new data (no bug)
Save path: /content/drive/MyDrive/EV_charging_project/datasets/valuation_new/

In [ ]:
!pip install implicit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=938107 sha256=8082eb19c15561cf9e2d4a1e46b3cd8130c56f9856dbfc6159f5872faa3fbeb0
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit


In [6]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares
from tqdm.auto import tqdm

# --- CẤU HÌNH ---
CSV_PATH     = "/content/drive/MyDrive/EV_charging_project/data_processing_files/processed_HCM_new.csv"
BASE_OUT_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new"

# Định nghĩa các kịch bản (Số User, Số Item mong muốn)
SCENARIOS = [
    (150, 180)
]

# Tham số ALS
FACTORS      = 32
ITERATIONS   = 20
REGULARIZATION = 0.01
ALPHA        = 40.0  # Alpha lớn hơn giúp tín hiệu rõ hơn khi data thưa

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def load_data(path):
    print(f"Loading raw data from {path}...")
    df = pd.read_csv(path, low_memory=False)
    df['thoi_gian_bat_dau'] = pd.to_datetime(df['thoi_gian_bat_dau'], errors='coerce')
    df = df.dropna(subset=['thoi_gian_bat_dau', 'so_khung', 'charger_id'])
    df['hour'] = df['thoi_gian_bat_dau'].dt.hour
    df['date'] = df['thoi_gian_bat_dau'].dt.date
    return df

def get_busiest_hour(df):
    # Tìm giờ cao điểm nhất trong toàn bộ dữ liệu để cố định
    hour_counts = df['hour'].value_counts()
    best_hour = hour_counts.idxmax()
    print(f"-> Giờ cao điểm nhất được chọn: {best_hour}:00 (Tổng {hour_counts.max()} sessions)")
    return best_hour

def process_scenario(df_full, target_users, target_items, best_hour):
    print(f"\n--- Xử lý kịch bản: {target_users} Users x {target_items} Items ---")

    # 1. Chỉ lấy dữ liệu trong giờ cao điểm
    df = df_full[df_full['hour'] == best_hour].copy()

    # 2. Chiến thuật tăng số lượng Items: Item = Charger + Day
    # Ta cần chọn bao nhiêu Charger và bao nhiêu Ngày để tích của chúng >= target_items?
    # Ưu tiên lấy nhiều Charger nhất có thể, sau đó mới tăng số ngày.

    unique_chargers = df['charger_id'].value_counts().index.tolist()
    n_chargers_avail = len(unique_chargers)

    n_chargers_to_take = min(n_chargers_avail, 50)
    n_days_needed = int(np.ceil(target_items / n_chargers_to_take))

    print(f"   Plan: Dùng {n_chargers_to_take} Chargers trong {n_days_needed} Ngày tốt nhất.")

    # 3. Lọc lấy Top Chargers và Top Days
    top_chargers = unique_chargers[:n_chargers_to_take]
    df = df[df['charger_id'].isin(top_chargers)]

    top_days = df['date'].value_counts().head(n_days_needed).index.tolist()
    df = df[df['date'].isin(top_days)].copy()

    # 4. Tạo Item ID mới
    charger_map = {c: i for i, c in enumerate(top_chargers)}
    day_map = {d: i for i, d in enumerate(sorted(top_days))} # Sort ngày để thứ tự logic

    df['c_idx'] = df['charger_id'].map(charger_map)
    df['d_idx'] = df['date'].map(day_map)

    df['item_id'] = df['d_idx'] * len(top_chargers) + df['c_idx']

    # Cắt bớt nếu lỡ tạo ra nhiều item hơn target (do làm tròn ngày)
    valid_items = list(range(target_items))
    df = df[df['item_id'].isin(valid_items)]

    actual_items = df['item_id'].nunique()

    # 5. Chọn Top Users trong không gian dữ liệu mới này
    top_users = df['so_khung'].value_counts().head(target_users).index.tolist()
    df = df[df['so_khung'].isin(top_users)].copy()

    user_map = {u: i for i, u in enumerate(top_users)}
    df['user_id'] = df['so_khung'].map(user_map)

    actual_users = df['user_id'].nunique()
    print(f"   -> Thực tế lọc được: {actual_users} Users và {actual_items} Items.")

    # 6. Tạo ma trận Valuation
    # Xây dựng Sparse Matrix cho ALS
    data = df['dien_nang_tieu_thu'].astype(np.float32).values
    rows = df['user_id'].values
    cols = df['item_id'].values

    R = sp.coo_matrix((data, (rows, cols)), shape=(target_users, target_items)).tocsr()

    # Train ALS
    model = AlternatingLeastSquares(factors=FACTORS, iterations=ITERATIONS, regularization=REGULARIZATION)
    model.fit(R * ALPHA)

    # 7. Áp dụng logic Sigmoid + Softmax (như code chuẩn trước đó)
    U_pos = sigmoid(model.user_factors)
    P_pos = sigmoid(model.item_factors)

    logits = U_pos @ P_pos.T

    # Zero-masking: Chỉ giữ giá trị ở những ô có observed items (items thực tế có trong top list)
    # Tuy nhiên, để mô phỏng thị trường "đầy đặn" hơn khi scale lớn,
    # ta có thể giữ nguyên dense matrix nhưng đảm bảo Softmax chuẩn.
    # Ở đây tôi dùng Softmax chuẩn hóa tổng = 10.

    logits -= logits.max(axis=1, keepdims=True)
    expV = np.exp(logits)
    V = 10.0 * expV / expV.sum(axis=1, keepdims=True)

    # Lưu file
    out_dir = os.path.join(BASE_OUT_DIR, f"val_{target_users}u_{target_items}i")
    os.makedirs(out_dir, exist_ok=True)

    np.save(f"{out_dir}/valuation_matrix.npy", V.astype(np.float32))
    # Lưu thêm factors nếu cần tái tạo
    np.save(f"{out_dir}/user_factors.npy", U_pos.astype(np.float32))
    np.save(f"{out_dir}/item_factors.npy", P_pos.astype(np.float32))

    # Lưu lookup để tra cứu ngược
    pd.Series(top_users).to_csv(f"{out_dir}/user_lookup.csv", header=['so_khung'], index=False)

    print(f"   ✓ Đã lưu kết quả vào: {out_dir}")
    print(f"     Shape ma trận V: {V.shape}")

def main():
    df = load_data(CSV_PATH)
    best_hour = get_busiest_hour(df)

    for n_u, n_i in SCENARIOS:
        process_scenario(df, n_u, n_i, best_hour)

if __name__ == "__main__":
    main()

Loading raw data from /content/drive/MyDrive/EV_charging_project/data_processing_files/processed_HCM_new.csv...
-> Giờ cao điểm nhất được chọn: 23:00 (Tổng 365 sessions)

--- Xử lý kịch bản: 150 Users x 180 Items ---
   Plan: Dùng 6 Chargers trong 30 Ngày tốt nhất.
   -> Thực tế lọc được: 150 Users và 172 Items.


  0%|          | 0/20 [00:00<?, ?it/s]

   ✓ Đã lưu kết quả vào: /content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_150u_180i
     Shape ma trận V: (150, 180)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
